# 🌌 MURE AGI: The Great Distillation & Alignment Pipeline
### Stage: Knowledge Pruning -> KL-Distillation -> DPO Alignment
**Created by Myo Min Aung**

This notebook implements a high-performance pipeline to create a specialized 3B parameter model from a 7B+ teacher, optimized for causal logic.

In [ ]:
# 1. Environment & Drive Verification\nfrom google.colab import drive\nimport os, sys\n\nif not os.path.exists("/content/drive/MyDrive"):\n    drive.mount("/content/drive", force_remount=True)\n\nBASE_DIR = "/content/drive/MyDrive/svo cc brain"\nos.makedirs(BASE_DIR, exist_ok=True)\nprint(f"✅ Workspace: {BASE_DIR}")\n

In [ ]:
# 2. Dependencies
!pip install -q torch transformers accelerate bitsandbytes peft datasets trl tqdm jsonlines
import torch, json, random
from tqdm.auto import tqdm
print(f"✅ Core ready on {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# 3. Robust Dataset Generation (5M Rules)
RULES_FILE = os.path.join(BASE_DIR, "rules_synthetic_5M.jsonl")
RULES_JSON_PATH = "/data/brain/rules.json"
TARGET_COUNT = 5_000_000

def load_rules():
    with open(RULES_JSON_PATH, 'r') as f:
        data = json.load(f)
        return data.get('causalMemory', data) if isinstance(data, dict) else data

rules = load_rules()
def check_progress():
    if not os.path.exists(RULES_FILE): return 0
    with open(RULES_FILE, "rb") as f:
        return sum(1 for _ in f)

current_rules = check_progress()
print(f"📊 Current Progress: {current_rules:,} / {TARGET_COUNT:,}")

if current_rules < TARGET_COUNT:
    print("⚡ Accelerating Generation...")
    TEMPLATES_QA = [
        ("What happens when {cause}?", "When {cause}, {effect}. (Confidence: {s:.2f})"),
        ("What is the effect of {cause}?", "The effect is {effect}. (Strength: {s:.2f})"),
        ("{cause} ဆိုတာ ဘာကိုဖြစ်စေသလဲ?", "{cause} ကြောင့် {effect} ဖြစ်ပေါ်သည်။"),
    ]
    with open(RULES_FILE, "a", encoding="utf-8") as f:
        for _ in tqdm(range(TARGET_COUNT - current_rules), desc="Building Mind Space"):
            rule = random.choice(rules)
            tmpl_q, tmpl_a = random.choice(TEMPLATES_QA)
            s = rule.get("strength", 0.8)
            item = {
                "cause": rule["cause"], "effect": rule["effect"],
                "strength": s,
                "instruction": tmpl_q.format(cause=rule["cause"], effect=rule["effect"]),
                "output": tmpl_a.format(cause=rule["cause"], effect=rule["effect"], s=s)
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
print("✅ Knowledge Core Solidified.")

In [ ]:
# 4. Teacher (7B) & Student (3B) Initialization\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nimport torch\n\nteacher_id = "Qwen/Qwen2.5-7B-Instruct"\nstudent_id = "Qwen/Qwen2.5-3B-Instruct"\n\nprint(f"🧠 Loading Teacher: {teacher_id} (4-bit)")\nbnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)\ntokenizer = AutoTokenizer.from_pretrained(teacher_id)\ntokenizer.pad_token = tokenizer.eos_token\n\nteacher = AutoModelForCausalLM.from_pretrained(teacher_id, quantization_config=bnb_config, device_map="auto")\n\nprint(f"👶 Loading Student: {student_id} (4-bit)")\nstudent_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)\nstudent = AutoModelForCausalLM.from_pretrained(student_id, quantization_config=student_bnb, device_map="auto")\nprint("✅ Brains Synchronized.")\n

In [ ]:
# 5. Knowledge Distillation Engine (High-Performance KL-D)
import os, torch, json
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

class HighSpeedDataset(Dataset):
    def __init__(self, path, tk, limit=500000):
        self.examples = []
        if os.path.exists(path):
            with open(path, "r") as f:
                for i, line in enumerate(f):
                    if i >= limit: break
                    item = json.loads(line)
                    self.examples.append(f"Cause: {item[\"cause\"]} -> Effect: {item[\"effect\"]}")
        self.tk = tk
    def __len__(self): return len(self.examples)
    def __getitem__(self, i):
        return self.tk(self.examples[i], truncation=True, padding=\"max_length\", max_length=128, return_tensors=\"pt\")

loader = DataLoader(HighSpeedDataset(RULES_FILE, tokenizer), batch_size=8, shuffle=True)
opt = torch.optim.AdamW(student.parameters(), lr=3e-5)
scaler = torch.cuda.amp.GradScaler()

for batch in tqdm(loader, desc="Distilling Knowledge (KL-Loss)"):
    ids = batch['input_ids'].squeeze(1).cuda()
    
    with torch.cuda.amp.autocast():
        with torch.no_grad(): t_logits = teacher(ids).logits
        s_logits = student(ids).logits
        
        B, T, V = s_logits.size()
        vm = min(s_logits.size(-1), t_logits.size(-1))
        
        s_flat = F.log_softmax(s_logits[..., :vm].reshape(-1, vm) / 2.0, dim=-1)
        t_flat = F.softmax(t_logits[..., :vm].reshape(-1, vm) / 2.0, dim=-1)
        
        loss = F.kl_div(s_flat, t_flat, reduction="batchmean") * 4.0
    
    if torch.cuda.is_bf16_supported():
        loss.backward(); opt.step(); opt.zero_grad()
    else:
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad()


In [ ]:
# 6. DPO Alignment (Direct Preference Optimization)
print("⚖️ Starting DPO Alignment...")
from trl import DPOConfig, DPOTrainer
from datasets import Dataset as HFDataset
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj","k_proj","v_proj","o_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
student_dpo = get_peft_model(student, lora_cfg)

# Load preference data properly
dpo_ds = load_dataset("json", data_files=os.path.join(BASE_DIR, "rules.json"), split="train") # Need to format this as DPO!

dpo_trainer = DPOTrainer(
    model=student_dpo, ref_model=None,
    args=DPOConfig(output_dir=os.path.join(BASE_DIR, "dpo_checkpoints"), num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=4), 
    train_dataset=dpo_ds,
    tokenizer=tokenizer,
)

dpo_trainer.train()


In [ ]:
# 7. Export MURE Brain
EXPORT_PATH = os.path.join(BASE_DIR, "MURE_Distilled_3B")
student.save_pretrained(EXPORT_PATH)
tokenizer.save_pretrained(EXPORT_PATH)
print(f"⭐ Distilled Brain Exported to: {EXPORT_PATH}")